In [1]:
# ============================================================
# Прогноз инфляции по её прошлым значениям
# ============================================================

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Сортировка обязательна: иначе lag_1 возьмёт значение «из будущего»
df = pd.read_csv("inflation_history.csv", parse_dates=["period_date"])
df = df.sort_values("period_date").reset_index(drop=True)

print("Размер:", df.shape)
print(df.head())
print(df.dtypes)

# Инфляция инерционна — прошлые месяцы сильно связаны с текущим
df["lag_1"] = df["inflation_percent"].shift(1)
df["lag_2"] = df["inflation_percent"].shift(2)
df["lag_3"] = df["inflation_percent"].shift(3)
# Месяц как признак: у инфляции есть сезонность (тарифы, продукты)
df["month"] = df["period_date"].dt.month

# Первые строки без истории обучить нельзя
df_model = df.dropna(subset=["lag_1", "lag_2", "lag_3", "inflation_percent"]).copy()
print(df_model.loc[:5, ["inflation_percent", "lag_1", "lag_2", "lag_3"]])

# Split по времени, не случайно: иначе модель увидит будущее (утечка)
features = ["lag_1", "lag_2", "lag_3", "month"]
target = "inflation_percent"

split_idx = int(len(df_model) * 0.8)
train = df_model.iloc[:split_idx]
test  = df_model.iloc[split_idx:]

X_train, y_train = train[features], train[target]
X_test,  y_test  = test[features],  test[target]

print(f"Train: {len(train)} строк, Test: {len(test)} строк")
print(f"Train период: {train['period_date'].min().date()} — {train['period_date'].max().date()}")
print(f"Test  период: {test['period_date'].min().date()} — {test['period_date'].max().date()}")

# LinearRegression как baseline: интерпретируемые коэффициенты, быстро
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

# MAE в единицах target, RMSE штрафует крупные промахи, R² — доля объяснённой дисперсии
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("\n=== Качество на тесте ===")
print(f"MAE : {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2  : {r2:.4f}")

# Сравнение факт vs прогноз — видно, где модель промахивается
result = pd.DataFrame({
    "period_date": test["period_date"].values,
    "actual": y_test.values,
    "predicted": y_pred,
})
result["error"] = result["actual"] - result["predicted"]
print("\nПоследние 10 прогнозов:")
print(result.tail(10).to_string(index=False))

# Коэффициенты: проверить знаки и отсутствие хаоса
coefs = pd.DataFrame({
    "feature": features,
    "coef": model.coef_,
}).sort_values("coef", key=abs, ascending=False)
print("\nКоэффициенты:")
print(coefs.to_string(index=False))
print(f"Intercept: {model.intercept_:.4f}")

Размер: (163, 3)
  period_date  inflation_percent                     created_at
0  2013-01-01               7.07  2026-08-22 17:36:31.760 +0300
1  2013-02-01               7.28  2026-08-22 17:36:31.760 +0300
2  2013-03-01               7.02  2026-08-22 17:36:31.760 +0300
3  2013-04-01               7.24  2026-08-22 17:36:31.760 +0300
4  2013-05-01               7.39  2026-08-22 17:36:31.760 +0300
period_date          datetime64[us]
inflation_percent           float64
created_at                      str
dtype: object
   inflation_percent  lag_1  lag_2  lag_3
3               7.24   7.02   7.28   7.07
4               7.39   7.24   7.02   7.28
5               6.89   7.39   7.24   7.02
Train: 128 строк, Test: 32 строк
Train период: 2013-04-01 — 2023-11-01
Test  период: 2023-12-01 — 2026-07-01

=== Качество на тесте ===
MAE : 0.3193
RMSE: 0.3868
R2  : 0.9350

Последние 10 прогнозов:
period_date  actual  predicted     error
 2025-10-01    7.71   7.882469 -0.172469
 2025-11-01    6.64   7.579

In [2]:
# Baseline «инфляция не изменится» — самая простая модель, которую надо побить
naive_pred = test["lag_1"].values
naive_mae  = mean_absolute_error(y_test, naive_pred)
naive_rmse = np.sqrt(mean_squared_error(y_test, naive_pred))
naive_r2   = r2_score(y_test, naive_pred)

print(f"Naive MAE : {naive_mae:.4f}  (модель: {mae:.4f})")
print(f"Naive RMSE: {naive_rmse:.4f}  (модель: {rmse:.4f})")
print(f"Naive R2  : {naive_r2:.4f}  (модель: {r2:.4f})")

Naive MAE : 0.3363  (модель: 0.3193)
Naive RMSE: 0.4324  (модель: 0.3868)
Naive R2  : 0.9187  (модель: 0.9350)
